In [ ]:
from utils.constants import REGIONS, PARTY_COLUMNS_2021, GPT_41_NANO, GPT_4o_MINI, GPT_41, GPT_4o
from utils.io_results import results_to_dataframe
from utils.voting import VotingProbabilities
from utils.aggregation import aggregate_results_from_df
from utils.metrics import load_actual_results
from utils.respondent_description import decline_region_to_locative
from utils.client import VotingProcessor
from utils import visualization
import numpy as np
import pandas as pd

In [ ]:
# MODEL = GPT_41_NANO
# MODEL = GPT_4o_MINI
# MODEL = GPT_41
MODEL = GPT_4o

processor = VotingProcessor()

In [3]:
def decline_region_to_Genitiv(region_name):
    if 'Česko' in region_name:
        return 'Česka'
    # Handles special case for Prague 'Hlavní město Praha'
    if 'Praha' in region_name:
        return 'Prahy'

    # General grammar rules for other regions
    declined_name = region_name.replace('raj', 'raje')  # Covers Kraj and kraj
    if 'ký' in declined_name:
        declined_name = declined_name.replace('ký', 'kého')

    return declined_name

In [ ]:
def citizen_prompt(region):
    return "Jsem obyvatel/ka "+ decline_region_to_Genitiv(region) + ". Ve volbách do poslanecké sněmovny v roce 2021 jsem volil:"
def direct_prompt(region):
    return "Účast a výsledek voleb do poslanecké sněmovny v "+ decline_region_to_locative(region)+ " v roce 2021 budou:"

In [5]:
print(citizen_prompt("Česko"))
print(direct_prompt("Česko"))

Jsem obyvatel/ka Česka. Ve volbách do poslanecké sněmovny v roce 2021 jsem volil:
Účast a výsledek voleb do poslanecké sněmovny v Česku v roce 2021 budou:


In [6]:
n = 10
prompt = citizen_prompt
# prompt = direct_prompt

In [ ]:
latent_results = []
for region in ["Česko"] + REGIONS:
    voting_results, skipped = processor.run(
        data=pd.DataFrame(np.random.rand(n, 1), columns=['Col']),
        prompt_creator= lambda res: prompt(region),
        response_model= VotingProbabilities,
        model= MODEL, temperature = 0,
    )
    results_df = results_to_dataframe(voting_results)
    model_latent_result = aggregate_results_from_df(results_df)
    model_latent_result['region'] = region
    latent_results.append(model_latent_result)

latent_results_df = pd.DataFrame(latent_results)
latent_results_df.to_csv(f"results/LatentPreferences_{MODEL}_{prompt.__name__}.csv")

In [ ]:
latent_results_df = pd.read_csv(f"results/LatentPreferences_{MODEL}_{prompt.__name__}.csv")
actual_results_df = load_actual_results()

for region in ["Česko"] + REGIONS:
     pred = latent_results_df[latent_results_df['region'] == region].iloc[0]
     actual = actual_results_df[actual_results_df["region"] == region].iloc[0]
     fig = visualization.visualize_comprehensive_results(
        pred,
        actual,
        PARTY_COLUMNS_2021,
        model_name=MODEL + " " + region, actual_is_claimed=False
    )

In [ ]:
cit = pd.read_csv(f"results/LatentPreferences_{MODEL}_{citizen_prompt.__name__}.csv")
dir = pd.read_csv(f"results/LatentPreferences_{MODEL}_{direct_prompt.__name__}.csv")


for region in ["Česko"] + REGIONS:
     pred = cit[cit['region'] == region].iloc[0]
     actual = dir[dir["region"] == region].iloc[0]
     fig = visualization.visualize_comprehensive_results(
        pred,
        actual,
        PARTY_COLUMNS_2021,
        model_name=MODEL + " " + region, actual_is_claimed=False
    )